# Phase 6 · Model 1 LoRMA

Fine-tune the Phase 1 model for legal summarization, evaluate its best checkpoint, and export test summaries.

## Required files and environment

- Attach `phase6_model1_lorma_assets.zip` or its extracted contents for the Phase 1 checkpoints and evaluator.
- Attach the official `Phase_6_Summ/` data, or enable Internet and set the `HF_TOKEN` secret to download it.
- Enable a GPU. To resume in a fresh session, restore the run folder and rerun setup before training.

In [1]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.34,<1', 'ijson>=3.3,<4', 'numpy>=1.26,<3',
    'pandas>=2.2,<3', 'matplotlib>=3.8,<4', 'tqdm>=4.66,<5',
    'rouge-score==0.1.2', 'bert-score==0.3.13', 'transformers>=4.45,<5'], check=True)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'huggingface-hub>=0.34,<1', 'ijson>=3.3,<4', 'numpy>=1.26,<3', 'pandas>=2.2,<3', 'matplotlib>=3.8,<4', 'tqdm>=4.66,<5', 'rouge-score==0.1.2', 'bert-score==0.3.13', 'transformers>=4.45,<5'], returncode=0)

In [2]:
import base64
import contextlib
import copy
import gc
import hashlib
import heapq
import html
import importlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace
from urllib.parse import quote

import ijson
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display, HTML, FileLink
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

PAD, UNK, BOS, EOS = range(4)
torch.set_num_threads(min(4, os.cpu_count() or 1))
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
for gpu in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu)
    print(f'GPU {gpu}: {props.name}, {props.total_memory / 2**30:.2f} GiB')


PyTorch: 2.11.0+cu128 | CUDA: True
GPU 0: Tesla T4, 14.56 GiB
GPU 1: Tesla T4, 14.56 GiB


## 2. Settings

Set the recipe before starting a fresh run. Signed checkpoints prevent silent changes on resumption. `DATA_REVISION=None` resolves and records a repository commit during download. If the repository changes between sessions, attach the first notebook's shared backup or set both notebooks to that recorded commit.

The initial recipe selects a head-and-tail context within the original 2,048-token model limit. Up to 1,536 source tokens and 384 reference-summary tokens are used, plus the prompt/control tokens. Longer references are truncated for training only; evaluation uses their complete reference summaries. The length report makes this visible. These are starting settings, not a claim that they are optimal.

Checkpoint selection uses BERTScore on at most 64 fixed held-out documents. Full held-out evaluation is a separate section. Both experiments use the same selection IDs. Training uses both GPUs; cached generation uses equal-length prompt batches on GPU 0. The decoder stops on EOS or its token limit; no fixed word-count requirement is imposed.


In [3]:
VERSION = 'phase6_summarization_v1'
DATA_VERSION = 'phase6_grouped_split_v1'
MODEL_NAME = 'model1'
METHOD = 'lorma'
RUN_NAME = 'model1_lorma'
UPLOAD_ZIP = 'phase6_model1_lorma_assets.zip'
EXPECTED_CHECKPOINT = 'cbdbdc8fff1d44d999a857997521135e56c6f92e7b53ec4eb1e72f514e0ce2f2'
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/phase6_model1_lorma')
ASSET_DIR = OUTPUT_ROOT / 'assets'
RUN_DIR = OUTPUT_ROOT / 'run'
for folder in (OUTPUT_ROOT, ASSET_DIR, RUN_DIR):
    folder.mkdir(parents=True, exist_ok=True)
MODEL_DIR = None
DATA_ROOT = None
SEARCH_ROOTS = []
REPO = 'Exploration-Lab/CS-781-Capstone'
DATA_REVISION = None
ID_FIELD, DOCUMENT_FIELD, SUMMARY_FIELD = 'id', 'document', 'summary'
SPLIT_SEED = 42
VALIDATION_FRACTION = 0.05
SELECTION_CASES = 64
PROMPT_PREFIX = 'Summarize the following legal document, preserving key facts, legal reasoning and the decision.\nDocument:\n'
PROMPT_SUFFIX = '\nSummary:\n'
RUN_SMALL_SAMPLE_CHECK = True
RUN_PER_DOCUMENT_BERTSCORE = True
RUN_GENERATION_CONTROLS = False
CONFIG = dict(seed=42, epochs=20, patience=3, source_tokens=1536, target_tokens=384,
    context_policy='head_tail', rank=8, alpha=16, adapter_dropout=0.05,
    lr=1e-4, min_lr_ratio=0.1, warmup_fraction=0.06, weight_decay=0.01,
    clip=1.0, effective_batch=8, micro_batch=1, activation_checkpointing=True,
    loss_chunk=128, log_every=20, save_every=100, max_hours=10.5,
    amp_initial_scale=1024.0, max_consecutive_amp_skips=12, capacity_headroom_gib=1.5,
    initial_generation_baseline=True)
DECODE = dict(temperature=0.0, top_p=0.95, max_new_tokens=384, min_new_tokens=32,
    no_repeat_ngram=4, repetition_penalty=1.1, batch_size=4, seed=42)
print(json.dumps(dict(run=RUN_NAME, training=CONFIG, decoding=DECODE), indent=2))


{
  "run": "model1_lorma",
  "training": {
    "seed": 42,
    "epochs": 20,
    "patience": 3,
    "source_tokens": 1536,
    "target_tokens": 384,
    "context_policy": "head_tail",
    "rank": 8,
    "alpha": 16,
    "adapter_dropout": 0.05,
    "lr": 0.0001,
    "min_lr_ratio": 0.1,
    "warmup_fraction": 0.06,
    "weight_decay": 0.01,
    "clip": 1.0,
    "effective_batch": 8,
    "micro_batch": 1,
    "activation_checkpointing": true,
    "loss_chunk": 128,
    "log_every": 20,
    "save_every": 100,
    "max_hours": 10.5,
    "amp_initial_scale": 1024.0,
    "max_consecutive_amp_skips": 12,
    "capacity_headroom_gib": 1.5,
    "initial_generation_baseline": true
  },
  "decoding": {
    "temperature": 0.0,
    "top_p": 0.95,
    "max_new_tokens": 384,
    "min_new_tokens": 32,
    "no_repeat_ngram": 4,
    "repetition_penalty": 1.1,
    "batch_size": 4,
    "seed": 42
  }
}


In [4]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)


def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 2**20), b''):
            digest.update(block)
    return digest.hexdigest()


def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':')).encode()).hexdigest()


def save_torch(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def unpack_inputs():
    roots = [INPUT_ROOT]
    for archive_path in sorted(INPUT_ROOT.rglob('*.zip')):
        if not archive_path.name.startswith(('phase6_', 'model3_lora_phase6_', 'model1_lorma_phase6_')):
            continue
        destination = OUTPUT_ROOT / 'imports' / archive_path.stem
        marker = destination / 'imported.json'
        identity = dict(bytes=archive_path.stat().st_size, path=str(archive_path),
                        modified=archive_path.stat().st_mtime_ns)
        if marker.is_file() and read_json(marker) == identity:
            roots.append(destination)
            continue
        with zipfile.ZipFile(archive_path) as archive:
            members = [item for item in archive.infolist() if not item.is_dir()
                       and Path(item.filename).parts
                       and (Path(item.filename).parts[0] == 'phase6_model_assets'
                            or Path(item.filename).parts[0].startswith('phase6_'))]
            if shutil.disk_usage(OUTPUT_ROOT).free < sum(item.file_size for item in members) + 256 * 2**20:
                raise OSError('Not enough working space to extract the attached Phase 6 ZIP.')
            for item in tqdm(members, desc=f'Import {archive_path.name}'):
                target = destination / item.filename
                if not target.resolve().is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe ZIP member.')
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst, length=8 * 2**20)
        write_json(marker, identity)
        roots.append(destination)
    return roots


def resolve_inputs():
    global SEARCH_ROOTS, MODEL_DIR, DATA_ROOT
    SEARCH_ROOTS = unpack_inputs()
    if MODEL_DIR is None:
        options = sorted(set(path.parent for root in SEARCH_ROOTS for path in root.rglob('run_config.json')
                             if all((path.parent / name).is_file() for name in ('best.pt', 'tokenizer.json'))))
        matching = [folder for folder in options if digest_file(folder / 'best.pt') == EXPECTED_CHECKPOINT]
        if not matching:
            raise FileNotFoundError(f'Attach {UPLOAD_ZIP}: the required Phase 1 checkpoint was not found.')
        MODEL_DIR = matching[0]
    MODEL_DIR = Path(MODEL_DIR)
    if digest_file(MODEL_DIR / 'best.pt') != EXPECTED_CHECKPOINT:
        raise ValueError('Wrong Phase 1 checkpoint. Do not use a Phase 5 classification checkpoint.')
    wanted = ['Train/train.json', 'Test/test.json']
    def valid(folder):
        return folder is not None and all((Path(folder) / name).is_file() for name in wanted)
    destination = ASSET_DIR / 'Phase_6_Summ'
    if valid(DATA_ROOT):
        source = Path(DATA_ROOT)
    elif valid(destination):
        source = destination
    else:
        options = sorted(set(path.parent.parent for root in SEARCH_ROOTS for path in root.rglob('train.json')
                             if path.parent.name == 'Train' and 'Phase_6_Summ' in path.parts and valid(path.parent.parent)))
        if len(options) > 1 and len({tuple(digest_file(folder / name) for name in wanted) for folder in options}) > 1:
            raise ValueError('Different Phase 6 datasets attached. Set DATA_ROOT explicitly.')
        if options:
            source = options[0]
        else:
            from huggingface_hub import HfApi, snapshot_download
            token = os.environ.get('HF_TOKEN')
            if not token:
                try:
                    from kaggle_secrets import UserSecretsClient
                    token = UserSecretsClient().get_secret('HF_TOKEN')
                except Exception:
                    token = None
            try:
                revision = DATA_REVISION or HfApi(token=token).dataset_info(REPO).sha
                snapshot = Path(snapshot_download(repo_id=REPO, repo_type='dataset', revision=revision,
                    allow_patterns=['Phase_6_Summ/**'], token=token))
            except Exception as error:
                raise RuntimeError('Enable Internet and grant this notebook access to the authorized HF_TOKEN secret, or attach Phase_6_Summ locally.') from error
            finally:
                token = None
            source = snapshot / 'Phase_6_Summ'
            write_json(ASSET_DIR / 'download_provenance.json', dict(repo=REPO, revision=revision))
    if not valid(source):
        raise FileNotFoundError(f'Missing official Phase_6_Summ files: {source}')
    for name in wanted:
        target = destination / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if source.resolve() != destination.resolve() and (not target.is_file() or digest_file(target) != digest_file(source / name)):
            shutil.copy2(source / name, target)
    DATA_ROOT = destination
    print('Phase 1 assets:', MODEL_DIR, flush=True)
    print('Official data:', DATA_ROOT, flush=True)


def restore_run():
    destination = RUN_DIR
    options = []
    for root in SEARCH_ROOTS:
        for path in root.rglob('signature.json'):
            if path.parent.resolve() == destination.resolve() or not (path.parent / 'latest.pt').is_file():
                continue
            try:
                if read_json(path).get('run_name') == RUN_NAME:
                    options.append(path.parent)
            except (ValueError, OSError):
                continue
    options = sorted(set(options))
    if not options or (destination / 'latest.pt').is_file():
        return
    if len({digest_file(folder / 'latest.pt') for folder in options}) != 1:
        raise ValueError('Attach one version of this run to resume.')
    files = [path for path in options[0].rglob('*') if path.is_file() and path.suffix != '.tmp']
    if shutil.disk_usage(OUTPUT_ROOT).free < sum(path.stat().st_size for path in files) + 256 * 2**20:
        raise OSError('Insufficient space to restore saved training state.')
    for path in files:
        target = destination / path.relative_to(options[0])
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and digest_file(target) != digest_file(path):
            raise ValueError(f'Existing file conflicts with attached backup: {target}')
        if not target.exists():
            shutil.copy2(path, target)
    print('Restored:', destination, flush=True)


In [6]:
resolve_inputs()


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

test.json: 0.00B [00:00, ?B/s]

Phase_6_Summ/Train/train.json:   0%|          | 0.00/211M [00:00<?, ?B/s]

Phase 1 assets: /kaggle/input/datasets/rishabhy26/assets/phase6_model_assets/model1_decoder
Official data: /kaggle/working/phase6_model1_lorma/assets/Phase_6_Summ


## 3. Original tokenizer and Phase 1 architecture


In [7]:
def split_bpe_pieces(text, max_chars=256):
    """Keep every character; separate words, punctuation and whitespace.

    An optional leading ASCII space can join a word or punctuation piece.
    Long pieces are split into bounded chunks to limit per-piece work.
    No normalization, strip(), split(), external tokenizer or regex package.
    """
    for match in re.finditer(' ?\\w+| ?[^\\w\\s]+|\\s+', text):
        piece = match.group(0)
        for start in range(0, len(piece), max_chars):
            yield piece[start:start + max_chars].encode('utf-8')


def apply_bpe_merges(piece, merge_ranks):
    """Apply learned merges by rank, breaking occurrence ties left-to-right.

    Token IDs 0..3 are reserved. Byte b always has ID b+4. A linked list
    and a heap avoid repeatedly scanning every adjacent pair in a piece.
    """
    ids = [byte + 4 for byte in piece]
    size = len(ids)
    if size < 2 or not merge_ranks:
        return tuple(ids)
    previous = [i - 1 for i in range(size)]
    following = [i + 1 for i in range(size)]
    following[-1] = -1
    active = [True] * size
    queue = []

    def offer(left):
        if left < 0 or not active[left]:
            return
        right = following[left]
        if right < 0:
            return
        pair = (ids[left], ids[right])
        rank = merge_ranks.get(pair)
        if rank is not None:
            heapq.heappush(queue, (rank, left, right, pair[0], pair[1]))
    for left in range(size - 1):
        offer(left)
    while queue:
        rank, left, right, a, b = heapq.heappop(queue)
        if not active[left] or not active[right] or following[left] != right:
            continue
        if ids[left] != a or ids[right] != b:
            continue
        ids[left] = 260 + rank
        active[right] = False
        following[left] = following[right]
        if following[right] >= 0:
            previous[following[right]] = left
        offer(previous[left])
        offer(left)
    result = []
    position = 0
    while position >= 0:
        result.append(ids[position])
        position = following[position]
    return tuple(result)


class ScratchByteBPE:
    """Our byte BPE vocabulary, encoder, decoder and JSON serialization.

    All base bytes and EVERY intermediate merge token are retained.
    Integer IDs identify symbols; concatenated decimal strings are never used.
    Literal '<s>' etc. in ordinary input are encoded as text, not control IDs.
    """
    FORMAT = 'handwritten-byte-bpe-v1'
    PATTERN = ' ?\\w+| ?[^\\w\\s]+|\\s+'

    def __init__(self, merges=None, max_piece_chars=256):
        self.specials = ['<pad>', '<unk>', '<s>', '</s>']
        self.max_piece_chars = max_piece_chars
        if not isinstance(max_piece_chars, int) or max_piece_chars < 1:
            raise ValueError('max_piece_chars must be a positive integer')
        self.merges = []
        self.merge_ranks = {}
        self.token_bytes = [None] * 4 + [bytes([b]) for b in range(256)]
        self._cache = {}
        for pair in merges or []:
            self.add_merge(tuple(pair))

    def add_merge(self, pair):
        if len(pair) != 2 or pair in self.merge_ranks:
            raise ValueError('Invalid or repeated merge pair')
        if any((type(token) is not int or not 4 <= token < len(self.token_bytes) for token in pair)):
            raise ValueError('A merge must reference earlier non-special token IDs')
        new_id = len(self.token_bytes)
        self.merge_ranks[pair] = len(self.merges)
        self.merges.append(pair)
        self.token_bytes.append(self.token_bytes[pair[0]] + self.token_bytes[pair[1]])
        self._cache.clear()
        return new_id

    def get_vocab_size(self):
        return len(self.token_bytes)

    def token_to_id(self, special):
        return self.specials.index(special) if special in self.specials else None

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError('encode expects a string')
        output = []
        for piece in split_bpe_pieces(text, self.max_piece_chars):
            encoded = self._cache.get(piece)
            if encoded is None:
                encoded = apply_bpe_merges(piece, self.merge_ranks)
                if len(self._cache) >= 65536:
                    self._cache.clear()
                self._cache[piece] = encoded
            output.extend(encoded)
        return SimpleNamespace(ids=output)

    def decode(self, ids, skip_special_tokens=True):
        chunks = []
        for value in ids:
            token = int(value)
            if not 0 <= token < len(self.token_bytes):
                raise ValueError(f'Unknown token ID: {token}')
            if token < 4:
                if not skip_special_tokens:
                    chunks.append(self.specials[token].encode('utf-8'))
            else:
                chunks.append(self.token_bytes[token])
        return b''.join(chunks).decode('utf-8', errors='replace')

    def save(self, path, training=None):
        payload = {'format': self.FORMAT, 'special_tokens': self.specials, 'byte_offset': 4, 'pattern': self.PATTERN, 'max_piece_chars': self.max_piece_chars, 'merges': self.merges, 'vocab_hex': [None if value is None else value.hex() for value in self.token_bytes]}
        if training is not None:
            payload['training'] = training
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
        temporary.replace(path)

    @classmethod
    def from_file(cls, path):
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        if payload.get('format') != cls.FORMAT:
            raise ValueError('This is not our handwritten BPE format. Use a fresh data directory and retrain the tokenizer/token arrays.')
        if payload.get('special_tokens') != ['<pad>', '<unk>', '<s>', '</s>']:
            raise ValueError('Special-token IDs do not match')
        if payload.get('byte_offset') != 4 or payload.get('pattern') != cls.PATTERN:
            raise ValueError('Byte mapping or splitting rules do not match')
        tokenizer = cls(payload['merges'], payload['max_piece_chars'])
        expected = [None if value is None else value.hex() for value in tokenizer.token_bytes]
        if payload.get('vocab_hex') != expected:
            raise ValueError('Vocabulary is inconsistent with the merge history')
        return tokenizer


In [8]:
class DecoderAttention(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim, self.dropout = (heads, width // heads, dropout)
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)

    def forward(self, x, past=None, cache=False):
        b, t, c = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.dim).unbind(2)
        q, k, v = (a.transpose(1, 2) for a in (q, k, v))
        if past is not None:
            if t != 1:
                raise ValueError('Cached decoding accepts one new token at a time')
            k, v = (torch.cat((past[0], k), 2), torch.cat((past[1], v), 2))
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=past is None)
        return (self.proj(y.transpose(1, 2).contiguous().view(b, t, c)), (k, v) if cache else None)


class Block(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = (nn.LayerNorm(width), nn.LayerNorm(width))
        self.attn = DecoderAttention(width, heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, past=None, cache=False):
        a, kv = self.attn(self.ln1(x), past, cache)
        x = x + self.drop(a)
        return (x + self.drop(self.mlp(self.ln2(x))), kv)


class DecoderLM(nn.Module):

    def __init__(self, vocab_size=32768, width=704, layers=16, heads=11, context=2048, dropout=0.0):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, width=width, layers=layers, heads=heads, context=context, dropout=dropout)
        self.token = nn.Embedding(vocab_size, width)
        self.position = nn.Embedding(context, width)
        self.blocks = nn.ModuleList([Block(width, heads, dropout) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(width)
        self.drop = nn.Dropout(dropout)
        self.gradient_checkpointing = True
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight, std=0.02 / math.sqrt(2 * layers))
            nn.init.normal_(block.mlp[2].weight, std=0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None, past=None, cache=False):
        offset = 0 if past is None else past[0][0].shape[2]
        if offset + ids.shape[1] > self.config['context']:
            raise ValueError('Context window exceeded')
        x = self.drop(self.token(ids) + self.position(torch.arange(offset, offset + ids.shape[1], device=ids.device)))
        kvs = []
        for i, block in enumerate(self.blocks):
            if self.training and self.gradient_checkpointing:
                x = checkpoint(lambda z, b=block: b(z)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    kvs.append(kv)
        x = self.final_norm(x)
        if targets is not None:
            flat, labels = (x.reshape(-1, x.shape[-1]), targets.reshape(-1))

            def chunk_loss(h, y):
                return F.cross_entropy(F.linear(h, self.token.weight).float(), y, ignore_index=-100, reduction='sum')
            loss = x.new_zeros((), dtype=torch.float32)
            for start in range(0, len(labels), 256):
                h, y = (flat[start:start + 256], labels[start:start + 256])
                loss = loss + (checkpoint(chunk_loss, h, y, use_reentrant=False) if self.training else chunk_loss(h, y))
            return loss
        return (F.linear(x[:, -1], self.token.weight), kvs)


In [9]:
MODEL_CLASS = DecoderLM
checkpoint_path = MODEL_DIR / 'best.pt'
tokenizer_path = MODEL_DIR / 'tokenizer.json'
saved_base = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
TOKENIZER = ScratchByteBPE.from_file(tokenizer_path)
BASE = dict(config=copy.deepcopy(saved_base['model_config']), state=saved_base['model'],
    checkpoint_sha256=digest_file(checkpoint_path), tokenizer_sha256=digest_file(tokenizer_path))
if BASE['config']['vocab_size'] != TOKENIZER.get_vocab_size():
    raise ValueError('Phase 1 tokenizer and model vocabulary differ.')
if saved_base.get('tokenizer_sha256') and saved_base['tokenizer_sha256'] != BASE['tokenizer_sha256']:
    raise ValueError('Phase 1 tokenizer hash mismatch.')
probe = MODEL_CLASS(**copy.deepcopy(BASE['config']))
probe.load_state_dict(BASE['state'], strict=True)
if not all(torch.isfinite(value).all().item() for value in BASE['state'].values() if value.is_floating_point()):
    raise FloatingPointError('Non-finite Phase 1 weights.')
base_info = dict(model=MODEL_NAME, config=BASE['config'], checkpoint=BASE['checkpoint_sha256'],
    tokenizer=BASE['tokenizer_sha256'], parameters=sum(p.numel() for p in probe.parameters()),
    source_metadata={key: saved_base[key] for key in ('step', 'tokens', 'total_data_tokens', 'predicted_tokens', 'val_loss') if key in saved_base})
write_json(ASSET_DIR / 'phase1_checkpoint.json', base_info)
print(json.dumps(base_info, indent=2))
del probe, saved_base
gc.collect()


{
  "model": "model1",
  "config": {
    "vocab_size": 32768,
    "width": 704,
    "layers": 16,
    "heads": 11,
    "context": 2048,
    "dropout": 0.0
  },
  "checkpoint": "cbdbdc8fff1d44d999a857997521135e56c6f92e7b53ec4eb1e72f514e0ce2f2",
  "tokenizer": "ce28d26edf4caa9358ffe59b573e362d9c05a90e881957fa0f2d054a5bfed152",
  "parameters": 119816576,
  "source_metadata": {
    "step": 6750,
    "tokens": 442365850,
    "val_loss": 2.0184897229003904
  }
}


0

## 4. Audit and document-level split

Exclude blank training references and keep identical documents in the same split.

In [10]:
def iter_records(path):
    with Path(path).open('rb') as handle:
        first = handle.read(1024).lstrip(b'\xef\xbb\xbf \n\r\t')[:1]
    with Path(path).open('rb') as handle:
        if first == b'[':
            yield from ijson.items(handle, 'item')
        elif first == b'{':
            for key, value in ijson.kvitems(handle, ''):
                if not isinstance(value, dict):
                    raise ValueError('Expected a list of records or a document-ID-to-record object.')
                yield dict(value, **({ID_FIELD: key} if ID_FIELD not in value else {}))
        else:
            for line in handle:
                if line.strip():
                    yield json.loads(line)


def text_field(record, key):
    value = record.get(key)
    if isinstance(value, list) and all(isinstance(part, str) for part in value):
        value = '\n'.join(value)
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f'Expected nonempty {key!r}; record keys: {list(record)}')
    return value


def audit_data():
    signature = dict(version=DATA_VERSION, train=digest_file(DATA_ROOT / 'Train/train.json'),
        test=digest_file(DATA_ROOT / 'Test/test.json'), seed=SPLIT_SEED, fraction=VALIDATION_FRACTION,
        selection_cases=SELECTION_CASES, fields=[ID_FIELD, DOCUMENT_FIELD, SUMMARY_FIELD])
    report_path = ASSET_DIR / 'data_audit.json'
    if report_path.is_file():
        saved = read_json(report_path)
        if saved['signature'] != signature:
            raise ValueError('Writable data preparation differs. Use a new OUTPUT_ROOT or restore its original data.')
        for name, digest in saved['prepared_sha256'].items():
            if not (ASSET_DIR / name).is_file() or digest_file(ASSET_DIR / name) != digest:
                raise ValueError(f'Prepared corpus/index changed: {name}')
        return read_json(ASSET_DIR / 'split.json'), saved
    test_hashes, seen_ids, rows, stats = set(), {}, {}, {}
    for split in ('test', 'train'):
        source = DATA_ROOT / ('Test/test.json' if split == 'test' else 'Train/train.json')
        corpus = ASSET_DIR / f'corpus_{split}.jsonl'
        temporary = corpus.with_suffix('.partial')
        rows[split] = []
        counts = Counter()
        seen_ids[split] = {}
        with temporary.open('wb') as outgoing:
            for record in tqdm(iter_records(source), desc=f'Audit {split}'):
                identifier = record.get(ID_FIELD)
                if isinstance(identifier, bool) or not isinstance(identifier, (str, int)) or not str(identifier).strip():
                    raise ValueError('Document IDs must be nonempty strings or integers.')
                identifier = str(identifier)
                document = text_field(record, DOCUMENT_FIELD)
                summary = text_field(record, SUMMARY_FIELD) if split == 'train' else None
                normalized = hashlib.sha256(' '.join(document.split()).encode()).hexdigest()
                identity = object_hash([document, summary])
                if identifier in seen_ids[split]:
                    if seen_ids[split][identifier] != identity:
                        raise ValueError(f'Conflicting repeated document ID in {split}: {identifier}')
                    counts['duplicate_ids'] += 1
                    continue
                seen_ids[split][identifier] = identity
                if split == 'train' and normalized in test_hashes:
                    counts['train_test_text_overlap_excluded'] += 1
                    continue
                if split == 'test':
                    test_hashes.add(normalized)
                case = dict(id=identifier, document=document)
                if summary is not None:
                    case['summary'] = summary
                encoded = (json.dumps(case, ensure_ascii=False) + '\n').encode()
                row = dict(id=identifier, normalized_hash=normalized, document_sha256=hashlib.sha256(document.encode()).hexdigest(),
                    summary_sha256=hashlib.sha256(summary.encode()).hexdigest() if summary is not None else None,
                    offset=outgoing.tell(), bytes=len(encoded), document_words=len(document.split()),
                    reference_words=len(summary.split()) if summary is not None else None)
                outgoing.write(encoded)
                rows[split].append(row)
                counts['retained_rows'] += 1
        temporary.replace(corpus)
        stats[split] = dict(counts)
    groups = sorted({row['normalized_hash'] for row in rows['train']},
                    key=lambda value: hashlib.sha256(f'{SPLIT_SEED}:{value}'.encode()).hexdigest())
    if len(groups) < 2:
        raise ValueError('Need at least two distinct training documents for a held-out split.')
    number = min(len(groups) - 1, max(1, round(len(groups) * VALIDATION_FRACTION)))
    heldout = set(groups[:number])
    index = dict(train=[row for row in rows['train'] if row['normalized_hash'] not in heldout],
                 val=[row for row in rows['train'] if row['normalized_hash'] in heldout], test=rows['test'])
    # Multiple summaries of identical text remain in the same split.
    assert not ({r['normalized_hash'] for r in index['train']} & {r['normalized_hash'] for r in index['val']})
    selected = sorted(index['val'], key=lambda r: hashlib.sha256(f'{SPLIT_SEED}:selection:{r["id"]}'.encode()).hexdigest())[:SELECTION_CASES]
    index['selection'] = selected
    write_json(ASSET_DIR / 'split.json', index)
    truth = {row['id']: read_case(row, 'val')['summary'] for row in index['val']}
    write_json(ASSET_DIR / 'val_truth.json', truth)
    write_json(ASSET_DIR / 'selection_truth.json', {row['id']: truth[row['id']] for row in selected})
    names = ['split.json', 'corpus_train.jsonl', 'corpus_test.jsonl', 'val_truth.json', 'selection_truth.json']
    report = dict(signature=signature, raw_statistics=stats, counts={s: len(v) for s, v in index.items()},
        document_groups=len(groups), split_sha256=object_hash(index),
        prepared_sha256={name: digest_file(ASSET_DIR / name) for name in names},
        policy='Whitespace-normalized document groups share a split; training copies of test documents are excluded; test summaries are never read.')
    write_json(report_path, report)
    return index, report


def read_case(row, split):
    path = ASSET_DIR / ('corpus_test.jsonl' if split == 'test' else 'corpus_train.jsonl')
    with path.open('rb') as handle:
        handle.seek(row['offset'])
        case = json.loads(handle.read(row['bytes']))
    if case['id'] != row['id']:
        raise ValueError('Prepared document offset/ID mismatch.')
    return case


def prepare_tokens():
    signature = dict(data=DATA_AUDIT['split_sha256'], tokenizer=BASE['tokenizer_sha256'], version=DATA_VERSION)
    folder = OUTPUT_ROOT / 'token_cache'
    roots = [root for search in SEARCH_ROOTS for root in search.rglob('token_cache') if root.is_dir()] + [folder]
    for candidate in roots:
        manifest_path = candidate / 'manifest.json'
        if not manifest_path.is_file():
            continue
        manifest = read_json(manifest_path)
        if manifest.get('signature') != signature:
            continue
        for name, digest in manifest['sha256'].items():
            if not (candidate / name).is_file() or digest_file(candidate / name) != digest:
                raise ValueError('Attached token cache is corrupt.')
        print('Reusing token cache:', candidate, flush=True)
        return candidate
    folder.mkdir(parents=True, exist_ok=True)
    index = {}
    for split in ('train', 'val', 'test'):
        progress_path = folder / f'{split}_progress.json'
        state = read_json(progress_path) if progress_path.is_file() else dict(signature=signature, rows=[], documents=0, summaries=0)
        if state['signature'] != signature:
            raise ValueError('Partial token cache belongs to a different split/tokenizer.')
        document_path, summary_path = folder / f'{split}_documents.bin', folder / f'{split}_summaries.bin'
        for path, count in ((document_path, state['documents']), (summary_path, state['summaries'])):
            if count and (not path.is_file() or path.stat().st_size < 2 * count):
                raise ValueError('Partial token cache is missing data.')
        with document_path.open('r+b' if document_path.exists() else 'w+b') as docs, summary_path.open('r+b' if summary_path.exists() else 'w+b') as sums:
            docs.truncate(2 * state['documents']); docs.seek(0, 2)
            sums.truncate(2 * state['summaries']); sums.seek(0, 2)
            for position, row in enumerate(tqdm(CASE_INDEX[split], desc=f'Tokenize {split}')):
                if position < len(state['rows']):
                    if state['rows'][position]['id'] != row['id']:
                        raise ValueError('Partial token cache order changed.')
                    continue
                case = read_case(row, split)
                document = TOKENIZER.encode(case['document']).ids
                summary = TOKENIZER.encode(case['summary']).ids if split != 'test' else []
                if not document or (split != 'test' and not summary):
                    raise ValueError(f'Empty tokenization: {row["id"]}')
                if min(document + summary) < 0 or max(document + summary) >= min(65536, BASE['config']['vocab_size']):
                    raise ValueError('Tokenizer/model vocabulary mismatch.')
                np.asarray(document, dtype='<u2').tofile(docs)
                np.asarray(summary, dtype='<u2').tofile(sums)
                state['rows'].append(dict(id=row['id'], document_start=state['documents'], document_length=len(document),
                                         summary_start=state['summaries'], summary_length=len(summary)))
                state['documents'] += len(document); state['summaries'] += len(summary)
                if (position + 1) % 50 == 0:
                    docs.flush(); sums.flush(); os.fsync(docs.fileno()); os.fsync(sums.fileno())
                    write_json(progress_path, state)
            docs.flush(); sums.flush()
            write_json(progress_path, state)
        index[split] = state['rows']
    write_json(folder / 'index.json', index)
    names = ['index.json'] + [f'{s}_{kind}.bin' for s in ('train', 'val', 'test') for kind in ('documents', 'summaries')]
    write_json(folder / 'manifest.json', dict(signature=signature, sha256={name: digest_file(folder / name) for name in names}))
    return folder


def source_tokens(tokens, cfg):
    budget = cfg['source_tokens']
    if len(tokens) <= budget:
        return list(tokens)
    gap = TOKENIZER.encode('\n[...]\n').ids
    if cfg['context_policy'] == 'head':
        return list(tokens[:budget])
    chunks = 2 if cfg['context_policy'] == 'head_tail' else 4 if cfg['context_policy'] == 'spread' else 0
    if not chunks or budget <= (chunks - 1) * len(gap) + chunks:
        raise ValueError('Invalid context selection or insufficient source budget.')
    available = budget - (chunks - 1) * len(gap)
    widths = [available // chunks + (i < available % chunks) for i in range(chunks)]
    starts = np.linspace(0, len(tokens) - max(widths), chunks).round().astype(int)
    selected = []
    for i, (start, width) in enumerate(zip(starts, widths)):
        if i:
            selected += gap
        if i == chunks - 1:
            start = len(tokens) - width
        selected += list(tokens[start:start + width])
    return selected


def prompt_tokens(document, cfg):
    prefix = [BOS] + TOKENIZER.encode(PROMPT_PREFIX).ids
    suffix = TOKENIZER.encode(PROMPT_SUFFIX).ids
    return prefix + source_tokens(document, cfg) + suffix


class SummaryDataset(Dataset):
    def __init__(self, split, cfg):
        self.split, self.cfg = split, cfg
        source_split = 'val' if split == 'selection' else split
        all_rows = TOKEN_INDEX[source_split]
        positions = {row['id']: i for i, row in enumerate(all_rows)}
        self.rows = [all_rows[positions[row['id']]] for row in CASE_INDEX[split]]
        self.documents = np.memmap(TOKEN_FOLDER / f'{source_split}_documents.bin', dtype='<u2', mode='r')
        summary_file = TOKEN_FOLDER / f'{source_split}_summaries.bin'
        self.summaries = np.memmap(summary_file, dtype='<u2', mode='r') if summary_file.stat().st_size else None
    def __len__(self):
        return len(self.rows)
    def document(self, i):
        row = self.rows[i]
        return self.documents[row['document_start']:row['document_start'] + row['document_length']].astype(np.int64).tolist()
    def prompt(self, i):
        return prompt_tokens(self.document(i), self.cfg)
    def target(self, i):
        row = self.rows[i]
        if self.summaries is None:
            return []
        start = row['summary_start']
        return self.summaries[start:start + min(row['summary_length'], self.cfg['target_tokens'])].astype(np.int64).tolist() + [EOS]
    def target_count(self, i):
        return min(self.rows[i]['summary_length'], self.cfg['target_tokens']) + 1
    def __getitem__(self, i):
        prompt, target = self.prompt(i), self.target(i)
        if not target:
            raise ValueError('The test dataset has no training targets.')
        combined = prompt + target
        x = torch.tensor(combined[:-1], dtype=torch.long)
        y = torch.tensor(combined[1:], dtype=torch.long)
        y[:len(prompt) - 1] = -100
        if len(x) > BASE['config']['context']:
            raise ValueError('Source and target exceed the original model context.')
        return x, y


def collate_summaries(samples):
    length = max(len(x) for x, _ in samples)
    ids = torch.full((len(samples), length), PAD, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    for i, (x, y) in enumerate(samples):
        ids[i, :len(x)], labels[i, :len(y)] = x, y
    return ids, labels


def token_report():
    report = {}
    for split in ('train', 'val', 'test'):
        rows = TOKEN_INDEX[split]
        lengths = np.asarray([r['document_length'] for r in rows])
        targets = np.asarray([r['summary_length'] for r in rows])
        gap_count = len(TOKENIZER.encode('\n[...]\n').ids) * (1 if CONFIG['context_policy'] == 'head_tail' else 3 if CONFIG['context_policy'] == 'spread' else 0)
        coverage = np.where(lengths <= CONFIG['source_tokens'], 1.0, (CONFIG['source_tokens'] - gap_count) / lengths)
        report[split] = dict(cases=len(rows), document_token_percentiles=np.percentile(lengths, [50, 90, 95, 99, 100]).tolist(),
            summary_token_percentiles=np.percentile(targets, [50, 90, 95, 99, 100]).tolist(),
            mean_document_coverage=float(coverage.mean()), truncated_summary_fraction=float((targets > CONFIG['target_tokens']).mean()) if split != 'test' else None)
    write_json(ASSET_DIR / 'token_length_and_coverage.json', report)
    print(json.dumps(report, indent=2), flush=True)
    return report


In [14]:
from pathlib import Path
from collections import Counter

# Preserve the original reader so rerunning this cell is safe.
_raw_phase6_reader = getattr(
    iter_records, "_phase6_original_reader", iter_records
)

_empty_summary_scan = Counter()
_exclusion_file = ASSET_DIR / "excluded_empty_summaries.json"


def iter_records(path, _reader=_raw_phase6_reader):
    is_training = (
        Path(path).resolve()
        == (DATA_ROOT / "Train/train.json").resolve()
    )
    excluded = []

    for record in _reader(path):
        if is_training:
            _empty_summary_scan["raw_records"] += 1
            summary = record.get(SUMMARY_FIELD)

            if isinstance(summary, str) and not summary.strip():
                excluded.append({
                    "id": str(record.get(ID_FIELD)),
                    "reason": "empty_reference_summary",
                })
                continue

        yield record

    if is_training:
        write_json(_exclusion_file, excluded)


iter_records._phase6_original_reader = _raw_phase6_reader

# Record the corrected preparation policy in cache signatures.
DATA_VERSION = "phase6_grouped_split_v2_blank_refs"

CASE_INDEX, DATA_AUDIT = audit_data()

excluded = read_json(_exclusion_file)
DATA_AUDIT["empty_summary_ids"] = [row["id"] for row in excluded]
DATA_AUDIT["raw_statistics"]["train"]["empty_summary_excluded"] = len(excluded)

if _empty_summary_scan["raw_records"]:
    DATA_AUDIT["raw_statistics"]["train"]["raw_records"] = (
        _empty_summary_scan["raw_records"]
    )

DATA_AUDIT["prepared_sha256"][_exclusion_file.name] = digest_file(
    _exclusion_file
)
DATA_AUDIT["empty_summary_policy"] = (
    "Blank training references are excluded and logged."
)
write_json(ASSET_DIR / "data_audit.json", DATA_AUDIT)

print("Excluded empty summaries:", DATA_AUDIT["empty_summary_ids"])
print("Final split counts:", DATA_AUDIT["counts"])
print("Paired split hash:", DATA_AUDIT["split_sha256"])

Audit test: 0it [00:00, ?it/s]

Audit train: 0it [00:00, ?it/s]

Excluded empty summaries: ['299', '4799']
Final split counts: {'train': 6678, 'val': 350, 'test': 100, 'selection': 64}
Paired split hash: 962a8eed648961f537d06cad815f30dfd66639b17a6073848b5d48ad9ea6f038


## 5. Shared token preparation and length report

Full document/reference token arrays are prepared once and reused on disk. Preparation resumes from saved offsets if interrupted. The token cache can be shared between these two notebooks because the existing Phase 1 tokenizers have the same hash. Attach a shared backup to reuse it, or let each session prepare independently.


In [15]:
TOKEN_FOLDER = prepare_tokens()
TOKEN_INDEX = read_json(TOKEN_FOLDER / 'index.json')
length_report = token_report()
maximum_prompt = len(prompt_tokens([4] * CONFIG['source_tokens'], CONFIG))
maximum_training = maximum_prompt + CONFIG['target_tokens']
print('Maximum prompt tokens:', maximum_prompt)
print('Maximum training input tokens:', maximum_training)
if maximum_training > BASE['config']['context'] or maximum_prompt + DECODE['max_new_tokens'] > BASE['config']['context']:
    raise ValueError('Reduce source/target/generation budgets before training; the original context limit is fixed.')


Tokenize train:   0%|          | 0/6678 [00:00<?, ?it/s]

Tokenize val:   0%|          | 0/350 [00:00<?, ?it/s]

Tokenize test:   0%|          | 0/100 [00:00<?, ?it/s]

{
  "train": {
    "cases": 6678,
    "document_token_percentiles": [
      3652.0,
      9109.6,
      13058.899999999998,
      27561.87999999989,
      162042.0
    ],
    "summary_token_percentiles": [
      749.0,
      1768.3000000000002,
      2393.1499999999996,
      5034.609999999997,
      32477.0
    ],
    "mean_document_coverage": 0.47858159417806395,
    "truncated_summary_fraction": 0.8622342018568434
  },
  "val": {
    "cases": 350,
    "document_token_percentiles": [
      3719.0,
      9203.500000000005,
      15593.05000000001,
      47417.06999999991,
      98150.0
    ],
    "summary_token_percentiles": [
      779.5,
      1928.0000000000005,
      2506.9500000000007,
      7685.379999999996,
      17199.0
    ],
    "mean_document_coverage": 0.47270124114977363,
    "truncated_summary_fraction": 0.8514285714285714
  },
  "test": {
    "cases": 100,
    "document_token_percentiles": [
      3761.5,
      10259.70000000001,
      16168.15,
      33900.92000000006

## 6. Summary model and adaptation

Input is BOS, instruction, selected document context, summary delimiter, reference summary and EOS. Loss is applied only to shifted reference-summary tokens and EOS. Document and padding tokens are masked. The original vocabulary and tied language-model output weights are retained; there is no classification head.

LoRA uses additive low-rank weight updates. LoRMA uses the paper's additive-identity inflation variant, **LoRMA+**, with a multiplicative transform of the frozen projection output. Both start as an identity change, train only their low-rank matrices, and target linear layers inside the Transformer blocks. Temporary inference copies merge these updates for generation; training checkpoints keep the adaptation matrices for resumption.

References: [LoRA](https://arxiv.org/abs/2106.09685), [LoRMA, Eq. 8](https://exploration-lab.github.io/LoRMA/static/paper.pdf).


In [16]:
class LowRankLinear(nn.Module):
    def __init__(self, base, method, rank, alpha, dropout):
        super().__init__()
        self.base, self.method = base, method
        base.requires_grad_(False)
        self.scale = alpha / rank
        self.drop = nn.Dropout(dropout)
        columns = base.in_features if method == 'lora' else base.out_features
        self.A = nn.Parameter(base.weight.new_empty(rank, columns))
        self.B = nn.Parameter(base.weight.new_zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        if self.method == 'lora':
            return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)
        if self.method == 'lorma':
            z = F.linear(x, self.base.weight)
            z = z + self.scale * F.linear(F.linear(self.drop(z), self.A), self.B)
            return z if self.base.bias is None else z + self.base.bias
        raise ValueError('Unknown adaptation method.')
    def merged(self):
        with torch.no_grad():
            delta = self.B.float() @ self.A.float()
            if self.method == 'lorma':
                delta = delta @ self.base.weight.float()
            self.base.weight.add_((self.scale * delta).to(self.base.weight.dtype))
        return self.base


def install_low_rank(module, cfg, prefix=''):
    names = []
    for name, child in list(module.named_children()):
        path = f'{prefix}.{name}' if prefix else name
        if isinstance(child, nn.Linear):
            setattr(module, name, LowRankLinear(child, METHOD, cfg['rank'], cfg['alpha'], cfg['adapter_dropout']))
            names.append(path)
        else:
            names += install_low_rank(child, cfg, path)
    return names


def merge_low_rank(module):
    for name, child in list(module.named_children()):
        if isinstance(child, LowRankLinear):
            setattr(module, name, child.merged())
        else:
            merge_low_rank(child)


class SummaryModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = copy.deepcopy(cfg)
        self.base = MODEL_CLASS(**copy.deepcopy(BASE['config']))
        self.base.load_state_dict(BASE['state'], strict=True)
        self.base.requires_grad_(False)
        self.adapted = install_low_rank(self.base.blocks, cfg)
        if not self.adapted:
            raise ValueError('No adaptation modules installed.')
        if MODEL_NAME == 'model1':
            self.base.gradient_checkpointing = cfg['activation_checkpointing']
        else:
            self.base.config['activation_checkpointing'] = cfg['activation_checkpointing']
    def hidden(self, ids, past=None, position_offset=0, cache=False):
        if MODEL_NAME == 'model3':
            return self.base.hidden(ids, past=past, position_offset=position_offset, cache=cache)
        length = ids.shape[1]
        if position_offset + length > self.base.config['context']:
            raise ValueError('Context exceeded.')
        positions = torch.arange(position_offset, position_offset + length, device=ids.device)
        x = self.base.drop(self.base.token(ids) + self.base.position(positions))
        caches = []
        for i, block in enumerate(self.base.blocks):
            if self.training and self.base.gradient_checkpointing:
                x = checkpoint(lambda z, layer=block: layer(z)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    caches.append(kv)
        return self.base.final_norm(x), caches
    def forward(self, ids, labels=None, past=None, position_offset=0, cache=False):
        hidden, caches = self.hidden(ids, past, position_offset, cache)
        if labels is None:
            return F.linear(hidden[:, -1], self.base.token.weight).float(), caches
        valid = labels.reshape(-1) >= 0
        selected = hidden.reshape(-1, hidden.shape[-1])[valid]
        targets = labels.reshape(-1)[valid]
        total = hidden.new_zeros((), dtype=torch.float32)
        def loss_chunk(h, y):
            return F.cross_entropy(F.linear(h, self.base.token.weight).float(), y, reduction='sum')
        for start in range(0, len(targets), self.cfg['loss_chunk']):
            h, y = selected[start:start + self.cfg['loss_chunk']], targets[start:start + self.cfg['loss_chunk']]
            total = total + (checkpoint(loss_chunk, h, y, use_reentrant=False)
                             if self.training and h.requires_grad else loss_chunk(h, y))
        return dict(loss_sum=total[None], tokens=total.new_tensor([len(targets)]))


def make_model(cfg=None, device='cpu', saved=None, merged=False):
    model = SummaryModel(cfg or CONFIG)
    if saved is not None:
        model.load_state_dict(saved, strict=True)
    if merged:
        merge_low_rank(model.base.blocks)
    return model.to(device)


def parameter_report(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
                adapted_modules=len(model.adapted))


def autocast_context(device):
    return torch.autocast('cuda', dtype=torch.float16) if str(device).startswith('cuda') else contextlib.nullcontext()


def sync_cuda():
    for gpu in range(torch.cuda.device_count()):
        torch.cuda.synchronize(gpu)


def make_optimizer(model, cfg):
    groups = [dict(params=[p for p in model.parameters() if p.requires_grad and (p.ndim > 1) == decay],
                   weight_decay=cfg['weight_decay'] if decay else 0.0) for decay in (True, False)]
    return torch.optim.AdamW([group for group in groups if group['params']], lr=cfg['lr'])


def make_scheduler(optimizer, cfg):
    total = math.ceil(len(CASE_INDEX['train']) / cfg['effective_batch']) * cfg['epochs']
    warmup = max(1, round(total * cfg['warmup_fraction']))
    def factor(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = min(1.0, (step - warmup) / max(1, total - warmup))
        return cfg['min_lr_ratio'] + (1 - cfg['min_lr_ratio']) * (1 + math.cos(math.pi * progress)) / 2
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def run_signature():
    return dict(version=VERSION, run_name=RUN_NAME, model=MODEL_NAME, method=METHOD,
        phase1_checkpoint=BASE['checkpoint_sha256'], tokenizer=BASE['tokenizer_sha256'],
        split=DATA_AUDIT['split_sha256'], token_cache=read_json(TOKEN_FOLDER / 'manifest.json'),
        config=CONFIG, decode=DECODE, prompt=[PROMPT_PREFIX, PROMPT_SUFFIX], torch=torch.__version__)


def self_checks():
    cfg = dict(CONFIG, activation_checkpointing=True, adapter_dropout=0.0)
    seed_everything(cfg['seed'])
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = make_model(cfg, device)
    for text in ('Section 42: भारत ₹100.', 'a  b\n c', '<s> literal text'):
        assert TOKENIZER.decode(TOKENIZER.encode(text).ids) == text
    ids = torch.tensor([TOKENIZER.encode('The court considered the evidence.').ids[:16]], dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        full = model(ids)[0]
        _, caches = model(ids[:, :-1], cache=True)
        incremental = model(ids[:, -1:], past=caches, position_offset=ids.shape[1] - 1, cache=True)[0]
        torch.testing.assert_close(full, incremental, rtol=3e-4, atol=3e-4)
        other = ids.clone(); other[:, -1] = (other[:, -1] + 1) % BASE['config']['vocab_size']
        torch.testing.assert_close(model.hidden(ids)[0][:, :-1], model.hidden(other)[0][:, :-1], rtol=0, atol=1e-6)
    sample = SummaryDataset('train', CONFIG)[0]
    x, y = collate_summaries([sample])
    model.train()
    losses = model(x.to(device), y.to(device))
    (losses['loss_sum'].sum() / losses['tokens'].sum()).backward()
    assert any(p.grad is not None and p.grad.abs().sum() > 0 for n, p in model.named_parameters() if n.endswith('.B'))
    assert all(p.grad is None for n, p in model.named_parameters() if '.A' not in n and '.B' not in n)
    assert int(losses['tokens'].sum()) == int((y >= 0).sum())
    assert all(torch.isfinite(p.grad).all().item() for p in model.parameters() if p.grad is not None)
    model.eval()
    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, LowRankLinear):
                module.B.normal_(std=0.001)
        before = model(ids)[0]
        merge_low_rank(model.base.blocks)
        torch.testing.assert_close(before, model(ids)[0], rtol=5e-4, atol=5e-4)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    report = dict(tokenizer_roundtrip=True, causality=True, incremental_cache=True,
                  summary_only_loss=True, frozen_base_gradients=True, adaptation_merge=True)
    write_json(ASSET_DIR / 'self_checks.json', report)
    print('Passed: tokenizer, causal masking, generation caches, adapter gradients, summary loss and merge.', flush=True)
    return report


## 7. Generation, official evaluation and training helpers

Generation uses existing per-layer caches, the original context bound, EOS, and explicit repetition controls. Completed batches are saved after every batch. A changed checkpoint, source policy, prompt or decode recipe needs a separate output directory; it cannot silently mix outputs.

The professor's evaluator is embedded unchanged below. It downloads BERTScore's scoring model through the metric library; that model is used only for evaluation, never as our summarizer or training teacher. Per-document BERTScore in the analysis uses the same metric settings.


In [17]:
def blocked_ngram_tokens(sequence, size):
    if size < 2 or len(sequence) < size - 1:
        return []
    prefix = tuple(sequence[-(size - 1):])
    return list({sequence[i + size - 1] for i in range(len(sequence) - size + 1)
                 if tuple(sequence[i:i + size - 1]) == prefix})


@torch.inference_mode()
def decode_batch(model, prompts, cfg, device, seed=42):
    if not prompts or len({len(p) for p in prompts}) != 1:
        raise ValueError('Generation batches must contain equal-length prompts.')
    length = len(prompts[0])
    if length + cfg['max_new_tokens'] > BASE['config']['context']:
        raise ValueError('Prompt plus generation budget exceeds the Phase 1 context.')
    generator = torch.Generator(device=device).manual_seed(seed)
    ids = torch.tensor(prompts, dtype=torch.long, device=device)
    model.eval()
    output = [[] for _ in prompts]
    reasons = ['length_limit'] * len(prompts)
    finished = [False] * len(prompts)
    past = None
    for step in range(cfg['max_new_tokens']):
        with autocast_context(device):
            logits, past = model(ids, past=past, position_offset=0 if step == 0 else length + step - 1, cache=True)
        scores = logits.float()
        if not torch.isfinite(scores).all().item():
            raise FloatingPointError('Non-finite generation logits.')
        scores[:, [PAD, UNK, BOS]] = float('-inf')
        if step < cfg['min_new_tokens']:
            scores[:, EOS] = float('-inf')
        for row, sequence in enumerate(output):
            if sequence and cfg['repetition_penalty'] != 1:
                positions = torch.tensor(sorted(set(sequence)), device=device)
                values = scores[row, positions]
                scores[row, positions] = torch.where(values < 0, values * cfg['repetition_penalty'], values / cfg['repetition_penalty'])
            blocked = blocked_ngram_tokens(sequence, cfg['no_repeat_ngram'])
            if blocked:
                scores[row, blocked] = float('-inf')
        if not torch.isfinite(scores).any(-1).all().item():
            raise RuntimeError('Decoding constraints removed every candidate token.')
        if cfg['temperature'] == 0:
            next_ids = scores.argmax(-1)
        else:
            sorted_scores, order = (scores / cfg['temperature']).sort(-1, descending=True)
            probabilities = sorted_scores.softmax(-1)
            removed = probabilities.cumsum(-1) - probabilities >= cfg['top_p']
            probabilities = probabilities.masked_fill(removed, 0)
            sampled = torch.multinomial(probabilities, 1, generator=generator)
            next_ids = order.gather(-1, sampled).squeeze(-1)
        for row, token in enumerate(next_ids.tolist()):
            if finished[row]:
                continue
            if token == EOS:
                finished[row] = True
                reasons[row] = 'eos'
            else:
                output[row].append(token)
        if all(finished):
            break
        ids = next_ids[:, None]
    records = []
    for sequence, reason in zip(output, reasons):
        summary = TOKENIZER.decode(sequence).strip()
        if not summary:
            raise RuntimeError('Generated an empty summary. Completed batches are saved; inspect this prompt before resuming.')
        records.append(dict(summary=summary, generated_tokens=len(sequence), stop_reason=reason))
    return records


def validate_prediction_rows(rows, expected_ids):
    if not isinstance(rows, list) or len(rows) != len(expected_ids):
        raise ValueError('Wrong number of generated summaries.')
    ids = []
    for row in rows:
        if set(row) != {'id', 'summary'} or not isinstance(row['id'], str) or not isinstance(row['summary'], str) or not row['summary'].strip():
            raise ValueError('Each output needs an exact ID and a nonempty summary string.')
        ids.append(row['id'])
    if len(set(ids)) != len(ids) or set(ids) != set(expected_ids):
        raise ValueError('Missing, duplicate or unknown prediction IDs.')


def generate_split(model, split, out, checkpoint_hash, policy=None, decode=None):
    cfg = dict(CONFIG, context_policy=policy or CONFIG['context_policy'])
    decode = copy.deepcopy(decode or DECODE)
    signature = dict(version=VERSION, checkpoint=checkpoint_hash, model=MODEL_NAME, method=METHOD,
        tokenizer=BASE['tokenizer_sha256'], cases=object_hash(CASE_INDEX[split]),
        context={key: cfg[key] for key in ('context_policy', 'source_tokens', 'target_tokens')},
        prompt=[PROMPT_PREFIX, PROMPT_SUFFIX], decode=decode, inference='merged_low_rank_v1')
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    metadata_path = out / 'generation.json'
    if metadata_path.is_file():
        if read_json(metadata_path) != signature:
            raise ValueError('Generation inputs changed. Use a different output folder.')
    elif any(out.iterdir()):
        raise ValueError('Generation folder is nonempty without metadata.')
    write_json(metadata_path, signature)
    details_path = out / 'details.json'
    completed = read_json(details_path) if details_path.is_file() else {}
    expected = [row['id'] for row in CASE_INDEX[split]]
    if set(completed) - set(expected):
        raise ValueError('Saved generation contains unexpected document IDs.')
    data = SummaryDataset(split, cfg)
    buckets = defaultdict(list)
    for i, row in enumerate(data.rows):
        if row['id'] not in completed:
            prompt = data.prompt(i)
            buckets[len(prompt)].append((i, prompt))
    device = next(model.parameters()).device
    started = time.perf_counter()
    for length in sorted(buckets, reverse=True):
        group = buckets[length]
        for start in range(0, len(group), decode['batch_size']):
            batch = group[start:start + decode['batch_size']]
            batch_started = time.perf_counter()
            result = decode_batch(model, [prompt for _, prompt in batch], decode, device,
                seed=decode['seed'] + int(object_hash([data.rows[i]['id'] for i, _ in batch])[:8], 16) % 1000000)
            batch_seconds = time.perf_counter() - batch_started
            for (i, prompt), generated in zip(batch, result):
                row = data.rows[i]
                completed[row['id']] = dict(**generated, prompt_tokens=len(prompt), document_tokens=row['document_length'],
                                           batch_seconds_per_document=batch_seconds / len(batch))
            write_json(details_path, completed)
            print(f'{split}: {len(completed)}/{len(data)} summaries', flush=True)
    rows = [dict(id=identifier, summary=completed[identifier]['summary']) for identifier in expected]
    validate_prediction_rows(rows, expected)
    write_json(out / 'prediction.json', rows)
    write_json(out / 'inference.json', dict(new_generation_seconds=time.perf_counter() - started,
        documents=len(rows), device=str(device), batch_size=decode['batch_size'],
        total_decode_seconds=sum(row.get('batch_seconds_per_document', 0) for row in completed.values()),
        checkpoint=checkpoint_hash, all_cached=not bool(buckets)))
    return out / 'prediction.json'


def run_evaluator(prediction_path, split, out):
    if split not in ('selection', 'val'):
        raise ValueError('Only held-out reference summaries may be evaluated.')
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    truth_path = ASSET_DIR / ('selection_truth.json' if split == 'selection' else 'val_truth.json')
    command = [sys.executable, str(EVALUATOR_PATH), '--truth', str(truth_path), '--prediction', str(prediction_path)]
    print(f'Professor evaluator: {split}', flush=True)
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout, flush=True)
    if result.stderr:
        print(result.stderr[-4000:], flush=True)
    (out / 'prof_evaluation.txt').write_text(result.stdout + '\n' + result.stderr, encoding='utf-8')
    write_json(out / 'evaluator_provenance.json', dict(script=digest_file(EVALUATOR_PATH),
        truth=digest_file(truth_path), prediction=digest_file(prediction_path), returncode=result.returncode))
    result.check_returncode()
    fields = {'bertscore_f1': r'BERTScore F1:\s*([\d.eE+-]+)', 'rouge1_f1': r'ROUGE-1:\s*([\d.eE+-]+)', 'rougel_f1': r'ROUGE-L:\s*([\d.eE+-]+)'}
    metrics = {}
    for key, pattern in fields.items():
        match = re.search(pattern, result.stdout)
        if not match or not math.isfinite(float(match.group(1))):
            raise RuntimeError('Could not read finite metrics from the official evaluator output.')
        metrics[key] = float(match.group(1))
    write_json(out / 'metrics.json', metrics)
    return metrics


def load_best_model():
    saved = torch.load(RUN_DIR / 'best.pt', map_location='cpu', weights_only=False)
    if saved['signature'] != run_signature():
        raise ValueError('Best checkpoint/data/settings signature mismatch.')
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = make_model(saved['runtime'], device, saved['model'], merged=True).eval()
    return model, saved


def export_predictions(split):
    model, saved = load_best_model()
    digest = digest_file(RUN_DIR / 'best.pt')
    out = RUN_DIR / ('validation' if split == 'val' else 'test') / object_hash(dict(checkpoint=digest, decode=DECODE))[:12]
    try:
        prediction = generate_split(model, split, out, digest)
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if split == 'val':
        metrics = run_evaluator(prediction, split, out)
        write_json(RUN_DIR / 'validation_latest.json', dict(folder=str(out), checkpoint=digest, metrics=metrics))
        return out
    archive_path = out / f'{RUN_NAME}_submission.zip'
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction, 'prediction.json')
    with zipfile.ZipFile(archive_path) as archive:
        assert archive.namelist() == ['prediction.json'] and archive.testzip() is None
    print('CodaBench submission:', archive_path, flush=True)
    # Submissions are small; a data link avoids the server-root 404 issue.
    encoded = base64.b64encode(archive_path.read_bytes()).decode()
    display(HTML(f'<a download="{archive_path.name}" href="data:application/zip;base64,{encoded}">Download {archive_path.name}</a>'))
    return archive_path


In [18]:
EVALUATOR_PATH = ASSET_DIR / 'evaluate_phase6.py'
EVALUATOR_SOURCE = '#!/usr/bin/env python3\n"""Local evaluator for Phase 6.\n\nRequires: rouge-score, bert-score, torch.\nTruth JSON: {"doc_id":"reference summary", ...}\nPrediction JSON: either {"doc_id":"summary", ...} or [{"id":"...","summary":"..."}, ...]\n"""\nimport argparse,json,sys\ntry:\n    from rouge_score import tokenizers, rouge_scorer\n    import bert_score\nexcept ImportError as e:\n    print(\'Missing dependency. Install requirements.txt first.\', file=sys.stderr); raise\n\ndef load(path):\n    with open(path,encoding=\'utf-8\') as f: return json.load(f)\n\ndef lcs(a,b):\n    if len(a)>len(b): a,b=b,a\n    masks={}\n    for i,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<i)\n    s=0\n    for t in b:\n        m=masks.get(t,0); x=m|s; y=(s<<1)|1; s=x & ~(x-y)\n    return s.bit_count() if hasattr(int,\'bit_count\') else bin(s).count(\'1\')\n\ndef read_pred(x):\n    if isinstance(x,dict): out=x\n    elif isinstance(x,list):\n        out={}\n        for r in x:\n            if not isinstance(r,dict) or \'id\' not in r or \'summary\' not in r: raise ValueError(\'Each prediction item must contain id and summary.\')\n            if r[\'id\'] in out: raise ValueError(f\'Duplicate prediction ID: {r["id"]}\')\n            out[r[\'id\']]=r[\'summary\']\n    else: raise ValueError(\'Prediction must be a JSON object or list of objects.\')\n    for k,v in out.items():\n        if not isinstance(k,str) or not isinstance(v,str) or not v.strip(): raise ValueError(f\'Invalid summary for ID {k!r}.\')\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 6 summaries on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); a=ap.parse_args()\n    t=load(a.truth); p=read_pred(load(a.prediction))\n    if not isinstance(t,dict) or not t: raise ValueError(\'Truth must be a non-empty JSON object.\')\n    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")\n    preds=[p[k].strip() for k in t]; refs=[t[k].strip() for k in t]\n    scorer=rouge_scorer.RougeScorer([\'rouge1\'],use_stemmer=True); tok=tokenizers.DefaultTokenizer(use_stemmer=True)\n    r1=rl=0.0\n    for pr,rf in zip(preds,refs):\n        s=scorer.score(rf,pr); r1+=s[\'rouge1\'].fmeasure\n        pt=tok.tokenize(pr); rt=tok.tokenize(rf)\n        if not pt and not rt: x=1.0\n        elif not pt or not rt: x=0.0\n        else:\n            z=lcs(pt,rt); pp=z/len(pt); rr=z/len(rt); x=0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)\n        rl+=x\n    n=len(t); r1/=n; rl/=n\n    try:\n        import torch; device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    except ImportError: device=None\n    _,_,f=bert_score.score(preds,refs,lang=\'en\',batch_size=8,verbose=False,**({\'device\':device} if device else {}))\n    print(f\'Documents: {n}\'); print(f\'BERTScore F1: {float(f.mean().item()):.6f}\'); print(f\'ROUGE-1: {r1:.6f}\'); print(f\'ROUGE-L: {rl:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n'
EVALUATOR_PATH.write_text(EVALUATOR_SOURCE, encoding='utf-8')
print('Evaluator:', EVALUATOR_PATH)


Evaluator: /kaggle/working/phase6_model1_lorma/assets/evaluate_phase6.py


In [19]:
def hardware_signature():
    return dict(version=VERSION, checkpoint=BASE['checkpoint_sha256'], config=CONFIG,
                devices=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                torch=torch.__version__)


def capacity_check():
    if not torch.cuda.is_available():
        raise RuntimeError('Enable the Kaggle GPU for capacity testing and full training.')
    path = RUN_DIR / 'hardware.json'
    if path.is_file():
        saved = read_json(path)
        if saved['signature'] != hardware_signature():
            raise ValueError('Hardware/settings changed. Restore the original GPU setup or use a new run folder.')
        print('Saved runtime:', saved['selected'], flush=True)
        return saved['selected']
    maximum_length = len(prompt_tokens([4] * CONFIG['source_tokens'], CONFIG)) + CONFIG['target_tokens']
    if maximum_length > BASE['config']['context']:
        raise ValueError('Maximum source/target shape exceeds the Phase 1 context.')
    results = []
    for checkpointing in (True, False):
        for micro in (1, 2, 4, 8):
            batch_size = micro * torch.cuda.device_count()
            if batch_size > CONFIG['effective_batch'] or CONFIG['effective_batch'] % batch_size:
                continue
            cfg = dict(CONFIG, micro_batch=micro, activation_checkpointing=checkpointing)
            print(f'Testing {micro} cases/GPU; checkpointing={checkpointing}', flush=True)
            model = net = optimizer = scaler = ids = labels = loss = output = norm = None
            try:
                gc.collect(); torch.cuda.empty_cache()
                seed_everything(cfg['seed'])
                model = make_model(cfg, 'cuda:0').train()
                net = nn.DataParallel(model) if torch.cuda.device_count() > 1 else model
                optimizer = make_optimizer(model, cfg)
                scaler = torch.amp.GradScaler('cuda', init_scale=cfg['amp_initial_scale'])
                ids = torch.randint(4, BASE['config']['vocab_size'], (batch_size, maximum_length), device='cuda:0')
                labels = torch.full_like(ids, -100)
                labels[:, -(cfg['target_tokens'] + 1):] = ids[:, -(cfg['target_tokens'] + 1):]
                accumulation = cfg['effective_batch'] // batch_size
                denominator = cfg['effective_batch'] * (cfg['target_tokens'] + 1)
                for gpu in range(torch.cuda.device_count()):
                    torch.cuda.reset_peak_memory_stats(gpu)
                durations, retries = [], 0
                while len(durations) < 4:
                    optimizer.zero_grad(set_to_none=True)
                    sync_cuda(); started = time.perf_counter()
                    for _ in range(accumulation):
                        with autocast_context('cuda'):
                            output = net(ids, labels)
                            loss = output['loss_sum'].sum() / denominator
                        if not torch.isfinite(loss).item():
                            raise FloatingPointError('Non-finite capacity loss.')
                        scaler.scale(loss).backward()
                        del output, loss
                        output = loss = None
                    scaler.unscale_(optimizer)
                    norm = nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg['clip'])
                    if not torch.isfinite(norm).item():
                        retries += 1
                        scaler.update(new_scale=scaler.get_scale() / 2)
                        if retries >= cfg['max_consecutive_amp_skips']:
                            raise FloatingPointError('Repeated non-finite capacity gradients.')
                        continue
                    scaler.step(optimizer); scaler.update()
                    sync_cuda(); durations.append(time.perf_counter() - started)
                    retries = 0
                memory = []
                for gpu in range(torch.cuda.device_count()):
                    free, total = torch.cuda.mem_get_info(gpu)
                    peak = torch.cuda.max_memory_reserved(gpu)
                    memory.append(dict(gpu=gpu, reserved_gib=peak / 2**30,
                        headroom_gib=min(free, total - peak) / 2**30))
                passed = all(row['headroom_gib'] >= cfg['capacity_headroom_gib'] for row in memory)
                results.append(dict(micro_batch=micro, activation_checkpointing=checkpointing,
                    passed=passed, seconds_per_update=float(np.mean(durations[1:])), memory=memory))
            except torch.OutOfMemoryError:
                results.append(dict(micro_batch=micro, activation_checkpointing=checkpointing, passed=False, reason='CUDA OOM'))
            except FloatingPointError as error:
                results.append(dict(micro_batch=micro, activation_checkpointing=checkpointing, passed=False, reason=str(error)))
            finally:
                model = net = optimizer = scaler = ids = labels = loss = output = norm = None
                gc.collect(); torch.cuda.empty_cache()
            print(results[-1], flush=True)
    passing = [row for row in results if row['passed']]
    if not passing:
        write_json(path, dict(signature=hardware_signature(), results=results, selected=None))
        raise RuntimeError('No safe maximum-shape runtime passed. Inspect the results before training.')
    selected = min(passing, key=lambda row: row['seconds_per_update'])
    write_json(path, dict(signature=hardware_signature(), results=results, selected=selected))
    updates = math.ceil(len(CASE_INDEX['train']) / CONFIG['effective_batch'])
    print('Selected:', selected, flush=True)
    print(f'Estimated training-only epoch time: {selected["seconds_per_update"] * updates / 3600:.2f} hours. Generation/evaluation and checkpoints add time.', flush=True)
    return selected


def runtime_config(micro_batch=None, activation_checkpointing=None):
    saved = read_json(RUN_DIR / 'hardware.json')
    if saved['signature'] != hardware_signature() or not saved['selected']:
        raise ValueError('Run a passing capacity check on this hardware first.')
    cfg = dict(CONFIG, **{key: saved['selected'][key] for key in ('micro_batch', 'activation_checkpointing')})
    if micro_batch is not None:
        cfg['micro_batch'] = micro_batch
    if activation_checkpointing is not None:
        cfg['activation_checkpointing'] = activation_checkpointing
    if not isinstance(cfg['micro_batch'], int) or cfg['micro_batch'] < 1:
        raise ValueError('Invalid microbatch.')
    return cfg


@torch.no_grad()
def validation_nll(model, cfg, split='selection'):
    model.eval()
    dataset = SummaryDataset(split, cfg)
    net = nn.DataParallel(model) if torch.cuda.device_count() > 1 and next(model.parameters()).is_cuda else model
    total, tokens = 0.0, 0
    device = next(model.parameters()).device
    loader = DataLoader(dataset, batch_size=cfg['micro_batch'] * max(1, torch.cuda.device_count()),
        shuffle=False, collate_fn=collate_summaries, num_workers=0)
    for ids, labels in loader:
        with autocast_context(device):
            output = net(ids.to(device), labels.to(device))
        total += float(output['loss_sum'].sum())
        tokens += int(output['tokens'].sum())
    loss = total / max(1, tokens)
    return dict(summary_nll=loss, perplexity=math.exp(min(loss, 30)), target_tokens=tokens)


def small_sample_check(updates=50, examples=8):
    cfg = runtime_config()
    cfg['adapter_dropout'] = 0.0
    seed_everything(cfg['seed'])
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = make_model(cfg, device)
    net = nn.DataParallel(model) if torch.cuda.device_count() > 1 and device.startswith('cuda') else model
    optimizer = make_optimizer(model, cfg)
    scaler = torch.amp.GradScaler('cuda', enabled=device.startswith('cuda'), init_scale=cfg['amp_initial_scale'])
    data = SummaryDataset('train', cfg)
    chosen = np.random.default_rng(cfg['seed']).choice(len(data), min(examples, len(data)), replace=False).tolist()
    denominator = sum(data.target_count(i) for i in chosen)
    batch_size = cfg['micro_batch'] * max(1, torch.cuda.device_count())
    history, completed, retries = [], 0, 0
    try:
        while completed < updates:
            model.train(); optimizer.zero_grad(set_to_none=True)
            total = 0.0
            for begin in range(0, len(chosen), batch_size):
                ids, labels = collate_summaries([data[i] for i in chosen[begin:begin + batch_size]])
                with autocast_context(device):
                    output = net(ids.to(device), labels.to(device))
                    loss = output['loss_sum'].sum() / denominator
                if not torch.isfinite(loss).item():
                    raise FloatingPointError('Non-finite tiny-sample loss.')
                scaler.scale(loss).backward()
                total += float(loss.detach())
                del ids, labels, output, loss
            scaler.unscale_(optimizer)
            norm = nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg['clip'])
            if not torch.isfinite(norm).item():
                retries += 1
                if not device.startswith('cuda') or retries >= cfg['max_consecutive_amp_skips']:
                    raise FloatingPointError('Tiny-sample gradients failed.')
                scaler.update(new_scale=scaler.get_scale() / 2)
                continue
            scaler.step(optimizer); scaler.update()
            completed += 1; retries = 0
            if completed == 1 or completed % 10 == 0 or completed == updates:
                row = dict(update=completed, summary_nll=total, perplexity=math.exp(min(total, 30)))
                history.append(row); print(row, flush=True)
    finally:
        model = net = optimizer = scaler = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    report = dict(case_ids=[CASE_INDEX['train'][i]['id'] for i in chosen], history=history,
                  loss_decreased=history[-1]['summary_nll'] < history[0]['summary_nll'])
    write_json(RUN_DIR / 'small_sample_check.json', report)
    if not report['loss_decreased']:
        print('Tiny-sample loss did not decrease. Inspect this diagnostic before starting the long run.', flush=True)
    return report


def log_metric(row):
    with (RUN_DIR / 'metrics.jsonl').open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row) + '\n')
    print(row, flush=True)


def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])


def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'])
    if torch.cuda.is_available() and state['cuda']:
        if len(state['cuda']) != torch.cuda.device_count():
            raise ValueError('Resume with the same GPU count.')
        torch.cuda.set_rng_state_all(state['cuda'])


def check_resume_memory():
    if not torch.cuda.is_available():
        raise RuntimeError('Enable the GPU for this disposable resume check.')
    import traceback
    for name in ('last_exc', 'last_value'):
        error = getattr(sys, name, None)
        if isinstance(error, BaseException) and error.__traceback__:
            traceback.clear_frames(error.__traceback__)
    gc.collect(); torch.cuda.empty_cache()
    cfg = runtime_config(micro_batch=1, activation_checkpointing=True)
    path = RUN_DIR / 'latest.pt'
    saved = torch.load(path, map_location='cpu', weights_only=False)
    if saved['signature'] != run_signature():
        raise ValueError('Checkpoint/data signature mismatch.')
    model = net = optimizer = scaler = ids = labels = output = loss = None
    try:
        model = make_model(cfg, 'cuda:0', saved['model']).train()
        optimizer = make_optimizer(model, cfg)
        optimizer.load_state_dict(saved['optimizer'])
        scaler = torch.amp.GradScaler('cuda', init_scale=cfg['amp_initial_scale'])
        scaler.load_state_dict(saved['scaler'])
        saved = None
        net = nn.DataParallel(model) if torch.cuda.device_count() > 1 else model
        batch_size = min(torch.cuda.device_count(), cfg['effective_batch'])
        length = len(prompt_tokens([4] * cfg['source_tokens'], cfg)) + cfg['target_tokens']
        ids = torch.randint(4, BASE['config']['vocab_size'], (batch_size, length), device='cuda:0')
        labels = torch.full_like(ids, -100)
        labels[:, -(cfg['target_tokens'] + 1):] = ids[:, -(cfg['target_tokens'] + 1):]
        optimizer.zero_grad(set_to_none=True)
        for start in range(0, cfg['effective_batch'], batch_size):
            count = min(batch_size, cfg['effective_batch'] - start)
            with autocast_context('cuda'):
                output = net(ids[:count], labels[:count])
                loss = output['loss_sum'].sum() / (cfg['effective_batch'] * (cfg['target_tokens'] + 1))
            scaler.scale(loss).backward()
            del output, loss
            output = loss = None
        scaler.unscale_(optimizer)
        norm = nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg['clip'])
        if not torch.isfinite(norm).item():
            raise FloatingPointError('Resume stress-test gradients were non-finite; saved checkpoints were not changed.')
        scaler.step(optimizer); scaler.update(); sync_cuda()
        print('Passed maximum-shape resume at microbatch 1/GPU with checkpointing. Saved checkpoints were not modified.', flush=True)
    finally:
        model = net = optimizer = scaler = saved = ids = labels = output = loss = None
        gc.collect(); torch.cuda.empty_cache()


def train_run(additional_steps=None, micro_batch=None, activation_checkpointing=None):
    cfg = runtime_config(micro_batch, activation_checkpointing)
    signature = run_signature()
    signature_path = RUN_DIR / 'signature.json'
    if signature_path.is_file() and read_json(signature_path) != signature:
        raise ValueError('Run settings/data changed. Restore the original recipe or use a new run folder.')
    write_json(signature_path, signature)
    write_json(RUN_DIR / 'config.json', dict(run_name=RUN_NAME, model=MODEL_NAME, method=METHOD, config=CONFIG, decode=DECODE))
    write_json(RUN_DIR / 'execution.json', cfg)
    seed_everything(cfg['seed'])
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = make_model(cfg, device)
    parameters = parameter_report(model)
    print('Parameters:', parameters, flush=True)
    net = nn.DataParallel(model) if torch.cuda.device_count() > 1 and device.startswith('cuda') else model
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg)
    scaler = torch.amp.GradScaler('cuda', enabled=device.startswith('cuda'), init_scale=cfg['amp_initial_scale'])
    data = SummaryDataset('train', cfg)
    state = dict(epoch=0, position=0, step=0, best_bertscore=-1.0, bad_epochs=0, complete=False,
                 skipped=0, consecutive_amp_skips=0, initial_evaluated=False)
    latest = RUN_DIR / 'latest.pt'
    if latest.is_file():
        saved = torch.load(latest, map_location='cpu', weights_only=False)
        if saved['signature'] != signature:
            raise ValueError('Resume signature mismatch.')
        model.load_state_dict(saved['model'], strict=True)
        optimizer.load_state_dict(saved['optimizer']); scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler']); state = saved['progress']; restore_rng(saved['rng'])
        del saved
        print('Resumed:', state, flush=True)
    started = time.monotonic(); start_step = state['step']
    batch_size = cfg['micro_batch'] * max(1, torch.cuda.device_count())
    def cpu_state():
        return {key: value.detach().cpu() for key, value in model.state_dict().items()}
    def save_latest():
        save_torch(latest, dict(signature=signature, model=cpu_state(), optimizer=optimizer.state_dict(),
            scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), progress=copy.deepcopy(state),
            rng=rng_state(), runtime=cfg))
        write_json(RUN_DIR / 'progress.json', state)
    def evaluate(count_patience):
        optimizer.zero_grad(set_to_none=True)
        nll = validation_nll(model, cfg)
        eval_rng = rng_state()
        weights = cpu_state()
        temporary_model = None
        try:
            temporary_model = make_model(cfg, device, weights, merged=True).eval()
            out = RUN_DIR / 'selection' / f'step_{state["step"]:08d}'
            identity = object_hash(dict(signature=signature, step=state['step']))
            prediction = generate_split(temporary_model, 'selection', out, identity)
        finally:
            temporary_model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            restore_rng(eval_rng)
        metrics = run_evaluator(prediction, 'selection', out)
        improved = metrics['bertscore_f1'] > state['best_bertscore'] + 1e-8
        if improved:
            state['best_bertscore'] = metrics['bertscore_f1']; state['bad_epochs'] = 0
            save_torch(RUN_DIR / 'best.pt', dict(signature=signature, model=weights, runtime=cfg,
                step=state['step'], epoch=state['epoch'], metrics=metrics, parameters=parameters))
        elif count_patience:
            state['bad_epochs'] += 1
        log_metric(dict(event='validation', step=state['step'], epoch=state['epoch'],
            best_bertscore=state['best_bertscore'], bad_epochs=state['bad_epochs'], **nll, **metrics))
        model.train()
    try:
        if not state['initial_evaluated']:
            if CONFIG['initial_generation_baseline']:
                evaluate(False)
            state['initial_evaluated'] = True
            save_latest()
        while not state['complete'] and state['epoch'] < cfg['epochs']:
            order = np.random.default_rng(cfg['seed'] + state['epoch']).permutation(len(data))
            model.train()
            while state['position'] < len(order):
                chosen = order[state['position']:state['position'] + cfg['effective_batch']]
                denominator = sum(data.target_count(int(i)) for i in chosen)
                optimizer.zero_grad(set_to_none=True)
                update_loss = 0.0
                for begin in range(0, len(chosen), batch_size):
                    ids, labels = collate_summaries([data[int(i)] for i in chosen[begin:begin + batch_size]])
                    with autocast_context(device):
                        output = net(ids.to(device), labels.to(device))
                        loss = output['loss_sum'].sum() / denominator
                    if not torch.isfinite(loss).item():
                        raise FloatingPointError('Non-finite training loss; inspect this batch.')
                    scaler.scale(loss).backward()
                    update_loss += float(loss.detach())
                    del ids, labels, output, loss
                scaler.unscale_(optimizer)
                norm = nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg['clip'])
                if not torch.isfinite(norm).item():
                    state['skipped'] += 1; state['consecutive_amp_skips'] += 1
                    if not device.startswith('cuda'):
                        raise FloatingPointError('Non-finite CPU gradients.')
                    scaler.update(new_scale=scaler.get_scale() / 2)
                    print('AMP overflow; retrying the same cases at a lower loss scale.', flush=True)
                    if state['consecutive_amp_skips'] >= cfg['max_consecutive_amp_skips']:
                        save_latest()
                        raise FloatingPointError('Repeated AMP failures. Resume state saved; inspect before continuing.')
                    continue
                scaler.step(optimizer); scaler.update(); scheduler.step()
                state['consecutive_amp_skips'] = 0
                state['step'] += 1; state['position'] += len(chosen)
                if state['step'] == start_step + 1 or state['step'] % cfg['log_every'] == 0:
                    log_metric(dict(event='train', step=state['step'], epoch=state['epoch'], summary_nll=update_loss,
                        lr=optimizer.param_groups[0]['lr'], gradient_norm=float(norm), skipped=state['skipped'],
                        elapsed_minutes=(time.monotonic() - started) / 60))
                if state['step'] % cfg['save_every'] == 0:
                    save_latest()
                if (additional_steps is not None and state['step'] - start_step >= additional_steps) or time.monotonic() - started >= cfg['max_hours'] * 3600:
                    save_latest()
                    report = dict(**state, session_paused=True, session_seconds=time.monotonic() - started)
                    write_json(RUN_DIR / 'session_report.json', report)
                    print(report, flush=True)
                    return report
            state['epoch'] += 1; state['position'] = 0
            evaluate(True)
            state['complete'] = state['epoch'] >= cfg['epochs'] or state['bad_epochs'] >= cfg['patience']
            if state['complete']:
                state['stop_reason'] = 'epochs' if state['epoch'] >= cfg['epochs'] else 'early_stopping'
            save_latest()
        report = dict(**state, session_seconds=time.monotonic() - started, parameters=parameters)
        write_json(RUN_DIR / 'summary.json', report)
        print(report, flush=True)
        return report
    except KeyboardInterrupt:
        save_latest()
        print('Interrupted; latest.pt saved. Rerun the training cell to resume.', flush=True)
        raise
    finally:
        optimizer.zero_grad(set_to_none=True)
        model = net = optimizer = scheduler = scaler = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [20]:
restore_run()


## 8. Functional checks

The disposable check verifies tokenizer round trips, causality, full-prefix/incremental cache agreement, summary-only loss, frozen base weights, adaptation gradients and merged inference. The saved Phase 1 and training checkpoints are not modified.


In [21]:
check_report = self_checks()


Passed: tokenizer, causal masking, generation caches, adapter gradients, summary loss and merge.


## 9. GPU capacity and speed

The test uses the maximum configured source+summary length, complete accumulation and optimizer updates. It compares microbatches and activation checkpointing, catches OOM/non-finite gradients, checks memory headroom on both GPUs, and selects the fastest passing training runtime. The time estimate excludes generation, BERTScore, checkpointing and preparation.


In [22]:
capacity = capacity_check()


Testing 1 cases/GPU; checkpointing=True
{'micro_batch': 1, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 1.2453602703333597, 'memory': [{'gpu': 0, 'reserved_gib': 2.064453125, 'headroom_gib': 12.27813720703125}, {'gpu': 1, 'reserved_gib': 0.716796875, 'headroom_gib': 13.63555908203125}]}
Testing 2 cases/GPU; checkpointing=True
{'micro_batch': 2, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 1.0153070903333326, 'memory': [{'gpu': 0, 'reserved_gib': 2.0234375, 'headroom_gib': 12.31915283203125}, {'gpu': 1, 'reserved_gib': 0.8984375, 'headroom_gib': 13.45391845703125}]}
Testing 4 cases/GPU; checkpointing=True
{'micro_batch': 4, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 0.8928477383333302, 'memory': [{'gpu': 0, 'reserved_gib': 1.724609375, 'headroom_gib': 12.61798095703125}, {'gpu': 1, 'reserved_gib': 1.412109375, 'headroom_gib': 12.94024658203125}]}
Testing 1 cases/GPU; checkpointing=False
{'micro_batch': 1,

## 10. Small-sample learning check

A disposable model fits eight fixed training documents for 50 updates. Check that summary-token NLL falls before spending a full session. This is a learning diagnostic, not a validation score or hyperparameter search; the real run still starts from the original Phase 1 weights.


In [23]:
if RUN_SMALL_SAMPLE_CHECK:
    fit_check = small_sample_check()
    if not fit_check['loss_decreased']:
        raise RuntimeError('Inspect the small-sample learning check before running full training.')


{'update': 1, 'summary_nll': 2.18359375, 'perplexity': 8.878154865526355}
{'update': 10, 'summary_nll': 1.9124908447265625, 'perplexity': 6.769930660677199}
{'update': 20, 'summary_nll': 1.422287940979004, 'perplexity': 4.146596764142811}
{'update': 30, 'summary_nll': 0.8675400018692017, 'perplexity': 2.3810462736714655}
{'update': 40, 'summary_nll': 0.44712477922439575, 'perplexity': 1.5638094180390674}
{'update': 50, 'summary_nll': 0.19709588587284088, 'perplexity': 1.2178608107543578}


## 11. Train or resume

Rerun the training cell to continue from the latest checkpoint.

In [24]:
training_report = train_run()


Parameters: {'total': 121438592, 'trainable': 1622016, 'adapted_modules': 64}
selection: 4/64 summaries
selection: 8/64 summaries
selection: 12/64 summaries
selection: 16/64 summaries
selection: 20/64 summaries
selection: 24/64 summaries
selection: 28/64 summaries
selection: 32/64 summaries
selection: 36/64 summaries
selection: 40/64 summaries
selection: 44/64 summaries
selection: 48/64 summaries
selection: 52/64 summaries
selection: 56/64 summaries
selection: 57/64 summaries
selection: 58/64 summaries
selection: 59/64 summaries
selection: 60/64 summaries
selection: 61/64 summaries
selection: 62/64 summaries
selection: 63/64 summaries
selection: 64/64 summaries
Professor evaluator: selection
Documents: 64
BERTScore F1: 0.810517
ROUGE-1: 0.264833
ROUGE-L: 0.145906

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task t

## 12. Full held-out validation

This evaluates every document in the grouped held-out split, including the fixed selection subset. Full reference summaries are used. Generated summaries, stop reasons, lengths, IDs, exact checkpoint/decode provenance, official evaluator output and aggregate metrics are saved.


In [25]:
validation_folder = export_predictions('val')


val: 4/350 summaries
val: 8/350 summaries
val: 12/350 summaries
val: 16/350 summaries
val: 20/350 summaries
val: 24/350 summaries
val: 28/350 summaries
val: 32/350 summaries
val: 36/350 summaries
val: 40/350 summaries
val: 44/350 summaries
val: 48/350 summaries
val: 52/350 summaries
val: 56/350 summaries
val: 60/350 summaries
val: 64/350 summaries
val: 68/350 summaries
val: 72/350 summaries
val: 76/350 summaries
val: 80/350 summaries
val: 84/350 summaries
val: 88/350 summaries
val: 92/350 summaries
val: 96/350 summaries
val: 100/350 summaries
val: 104/350 summaries
val: 108/350 summaries
val: 112/350 summaries
val: 116/350 summaries
val: 120/350 summaries
val: 124/350 summaries
val: 128/350 summaries
val: 132/350 summaries
val: 136/350 summaries
val: 140/350 summaries
val: 144/350 summaries
val: 148/350 summaries
val: 152/350 summaries
val: 156/350 summaries
val: 160/350 summaries
val: 164/350 summaries
val: 168/350 summaries
val: 172/350 summaries
val: 176/350 summaries
val: 180/350 s

## 13. Analysis exports

The next cells save per-document BERTScore/ROUGE, source coverage, output/reference lengths, repeated n-grams, source overlap, length-group scores, training curves, parameter counts, hardware/timing and checkpoint size. Source overlap is only a descriptive measure; it does not establish factuality.

`manual_legal_review.csv` and `manual_review_documents.jsonl` retain a review sample with full source documents and reference/generated summaries. Assess facts, reasoning, decisions, unsupported claims and missing information yourself. These exports provide evidence for the report; they do not write conclusions automatically.


In [26]:
def repeated_ngram_fraction(words, size):
    grams = [tuple(words[i:i + size]) for i in range(max(0, len(words) - size + 1))]
    return (len(grams) - len(set(grams))) / max(1, len(grams))


def analyze_validation():
    pointer = read_json(RUN_DIR / 'validation_latest.json')
    if pointer['checkpoint'] != digest_file(RUN_DIR / 'best.pt'):
        raise ValueError('Validation belongs to an older best checkpoint. Rerun validation first.')
    folder = Path(pointer['folder'])
    rows = read_json(folder / 'prediction.json')
    detail = read_json(folder / 'details.json')
    truth = read_json(ASSET_DIR / 'val_truth.json')
    validate_prediction_rows(rows, truth)
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    output = RUN_DIR / 'analysis' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    output.mkdir(parents=True, exist_ok=False)
    case_rows = []
    metadata = {row['id']: row for row in TOKEN_INDEX['val']}
    case_metadata = {row['id']: row for row in CASE_INDEX['val']}
    for record in rows:
        identifier = record['id']
        predicted = record['summary']; reference = truth[identifier]
        words = predicted.split(); ref_words = reference.split()
        score = scorer.score(reference, predicted)
        token_data = metadata[identifier]
        original = read_case(case_metadata[identifier], 'val')['document']
        document_words = original.split()
        document_ngrams = {tuple(document_words[i:i + 3]) for i in range(max(0, len(document_words) - 2))}
        summary_ngrams = [tuple(words[i:i + 3]) for i in range(max(0, len(words) - 2))]
        gap_count = len(TOKENIZER.encode('\n[...]\n').ids) * (1 if CONFIG['context_policy'] == 'head_tail' else 3 if CONFIG['context_policy'] == 'spread' else 0)
        coverage = 1.0 if token_data['document_length'] <= CONFIG['source_tokens'] else (CONFIG['source_tokens'] - gap_count) / token_data['document_length']
        case_rows.append(dict(id=identifier, generated_summary=predicted, reference_summary=reference,
            generated_words=len(words), reference_words=len(ref_words),
            generated_tokens=detail[identifier]['generated_tokens'], stop_reason=detail[identifier]['stop_reason'],
            document_tokens=token_data['document_length'], reference_tokens=token_data['summary_length'],
            reference_truncated_for_training=token_data['summary_length'] > CONFIG['target_tokens'],
            selected_context_fraction=coverage,
            rouge1_f1=score['rouge1'].fmeasure, rougel_f1=score['rougeL'].fmeasure,
            repeated_3gram_fraction=repeated_ngram_fraction(words, 3), repeated_4gram_fraction=repeated_ngram_fraction(words, 4),
            source_overlap_3gram_fraction=sum(gram in document_ngrams for gram in summary_ngrams) / max(1, len(summary_ngrams))))
    frame = pd.DataFrame(case_rows)
    if RUN_PER_DOCUMENT_BERTSCORE:
        import bert_score
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        _, _, scores = bert_score.score(frame['generated_summary'].tolist(), frame['reference_summary'].tolist(),
                                        lang='en', batch_size=8, device=device, verbose=False)
        frame['bertscore_f1'] = scores.cpu().numpy()
        del scores
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    frame.to_csv(output / 'per_document.csv', index=False)
    lengths = pd.cut(frame['document_tokens'], bins=[0, 1536, 4096, 8192, np.inf], labels=['up_to_1536', '1537_to_4096', '4097_to_8192', 'over_8192'])
    frame.assign(length_group=lengths).groupby('length_group', observed=True).agg(
        documents=('id', 'count'), rouge1_f1=('rouge1_f1', 'mean'), rougel_f1=('rougel_f1', 'mean'),
        context_coverage=('selected_context_fraction', 'mean'), repetition=('repeated_4gram_fraction', 'mean')).to_csv(output / 'document_length_groups.csv')
    manual = frame.sort_values('rougel_f1').head(min(20, len(frame))).copy()
    for column in ('facts_preserved', 'reasoning_preserved', 'decision_preserved', 'unsupported_claims', 'missed_information', 'notes'):
        manual[column] = ''
    manual.to_csv(output / 'manual_legal_review.csv', index=False)
    with (output / 'manual_review_documents.jsonl').open('w', encoding='utf-8') as handle:
        for identifier in manual['id']:
            case = read_case(case_metadata[identifier], 'val')
            handle.write(json.dumps(dict(case, generated_summary=next(r['summary'] for r in rows if r['id'] == identifier)), ensure_ascii=False) + '\n')
    history_path = RUN_DIR / 'metrics.jsonl'
    history = [json.loads(line) for line in history_path.read_text().splitlines() if line.strip()] if history_path.is_file() else []
    pd.DataFrame(history).to_csv(output / 'training_history.csv', index=False)
    validation = pd.DataFrame([row for row in history if row.get('event') == 'validation'])
    if not validation.empty:
        figure, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].plot(validation['epoch'], validation['bertscore_f1'], marker='o'); axes[0].set(xlabel='Epoch', ylabel='Selection BERTScore F1')
        axes[1].plot(validation['epoch'], validation['summary_nll'], marker='o'); axes[1].set(xlabel='Epoch', ylabel='Selection summary NLL')
        figure.tight_layout(); figure.savefig(output / 'training_curves.png', dpi=150); plt.close(figure)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(frame['generated_words'], frame['rougel_f1'], s=8, alpha=0.5); axes[0].set(xlabel='Generated words', ylabel='ROUGE-L F1')
    axes[1].scatter(frame['selected_context_fraction'], frame['rouge1_f1'], s=8, alpha=0.5); axes[1].set(xlabel='Selected context fraction', ylabel='ROUGE-1 F1')
    figure.tight_layout(); figure.savefig(output / 'length_and_coverage.png', dpi=150); plt.close(figure)
    saved = torch.load(RUN_DIR / 'best.pt', map_location='cpu', weights_only=False)
    write_json(output / 'summary.json', dict(full_validation=pointer['metrics'], selection_metrics=saved['metrics'],
        parameters=saved['parameters'], checkpoint_bytes=(RUN_DIR / 'best.pt').stat().st_size,
        best_epoch=saved['epoch'], best_step=saved['step'], progress=read_json(RUN_DIR / 'progress.json'),
        inference=read_json(folder / 'inference.json'), hardware=read_json(RUN_DIR / 'hardware.json'),
        source_policy=CONFIG['context_policy'], decode=DECODE,
        legal_review='Complete manual_legal_review.csv yourself. Lexical overlap and ROUGE are not factuality judgments.',
        limitations=['One fixed selection subset is reused for checkpoint selection.',
                    'Reference targets may be truncated for training; full references are used for evaluation.',
                    'Source coverage is constrained by the original context length.',
                    'Per-document ROUGE is diagnostic; aggregate metrics come from the professor evaluator.']))
    del saved
    print(frame[['generated_words', 'reference_words', 'rouge1_f1', 'rougel_f1', 'repeated_4gram_fraction']].describe().to_string(), flush=True)
    print('Analysis saved:', output, flush=True)
    return output


def run_generation_control(context_policy=None, max_new_tokens=None):
    policy = context_policy or CONFIG['context_policy']
    decode = dict(DECODE)
    if max_new_tokens is not None:
        decode['max_new_tokens'] = max_new_tokens
        decode['min_new_tokens'] = min(decode['min_new_tokens'], max_new_tokens - 1)
    model, saved = load_best_model()
    digest = digest_file(RUN_DIR / 'best.pt')
    identity = object_hash(dict(checkpoint=digest, policy=policy, decode=decode))[:12]
    out = RUN_DIR / 'generation_controls' / identity
    try:
        prediction = generate_split(model, 'selection', out, digest, policy=policy, decode=decode)
    finally:
        del model, saved
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    metrics = run_evaluator(prediction, 'selection', out)
    write_json(out / 'interpretation.json', dict(context_policy=policy, decode=decode,
        note='Inference-only control on the same checkpoint. This is not a separately trained technique or evidence of causation.'))
    return metrics


In [27]:
analysis_folder = analyze_validation()


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


       generated_words  reference_words   rouge1_f1   rougel_f1  repeated_4gram_fraction
count       350.000000       350.000000  350.000000  350.000000               350.000000
mean        303.322857       917.780000    0.422026    0.234341                 0.000195
std          48.953592      1176.125773    0.125845    0.077452                 0.000846
min          93.000000        57.000000    0.032105    0.023649                 0.000000
25%         294.000000       416.250000    0.340580    0.184166                 0.000000
50%         321.000000       656.500000    0.435032    0.229374                 0.000000
75%         333.000000      1013.750000    0.517120    0.279581                 0.000000
max         363.000000     13861.000000    0.723861    0.536193                 0.008772
Analysis saved: /kaggle/working/phase6_model1_lorma/run/analysis/20261030_172822_007112


## 14. Official test submission

The test parser uses only IDs and documents. Generation resumes completed batches. The output validator requires exactly one nonempty summary per official ID. The small CodaBench ZIP contains `prediction.json` directly at its root; the link below downloads that submission, not the checkpoint backup.


In [28]:
submission_zip = export_predictions('test')


test: 4/100 summaries
test: 8/100 summaries
test: 12/100 summaries
test: 16/100 summaries
test: 20/100 summaries
test: 24/100 summaries
test: 28/100 summaries
test: 32/100 summaries
test: 36/100 summaries
test: 40/100 summaries
test: 44/100 summaries
test: 48/100 summaries
test: 52/100 summaries
test: 56/100 summaries
test: 60/100 summaries
test: 64/100 summaries
test: 68/100 summaries
test: 72/100 summaries
test: 76/100 summaries
test: 80/100 summaries
test: 84/100 summaries
test: 88/100 summaries
test: 92/100 summaries
test: 94/100 summaries
test: 95/100 summaries
test: 96/100 summaries
test: 97/100 summaries
test: 98/100 summaries
test: 99/100 summaries
test: 100/100 summaries
CodaBench submission: /kaggle/working/phase6_model1_lorma/run/test/756cf34628b3/model1_lorma_submission.zip


## 15. Optional generation controls

Leave this off until the compulsory runs finish. These cells compare head, head-and-tail and spread context, and a shorter generation limit, using the same trained checkpoint and fixed selection documents. They support the long-document/length analysis but do not count as new techniques or independent training experiments. Each variant has separate output provenance.


In [ ]:
if RUN_GENERATION_CONTROLS:
    controls = []
    for policy in ('head', 'head_tail', 'spread'):
        controls.append(dict(context_policy=policy, max_new_tokens=DECODE['max_new_tokens'], **run_generation_control(context_policy=policy)))
    controls.append(dict(context_policy=CONFIG['context_policy'], max_new_tokens=256, **run_generation_control(max_new_tokens=256)))
    pd.DataFrame(controls).to_csv(RUN_DIR / 'generation_controls/comparison.csv', index=False)
    display(pd.DataFrame(controls))
